### Quality check analysis of anndata object from re-mapped E-MTAB-8901 samples
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 2nd of December 2024
- **Last modified date:** 2nd of December 2024

This notebook contains such parts:
* Doublets score prediction with `scrublet`
* Calculating quality metrics with `sctk`
* Identify cells that pass QC on:
    * Individual cell level
    * QC cluster level
    * Both individual cell and cluster levels
* Visualize quality metrics before filtering
* Filter out cells that do not pass QC
* Visualize quality metrics after filtering
* Identify sex covariates
* Calculate cell cycle score

### Import packages

In [ ]:
import scanpy as sc
import scrublet as scr
import numpy as np
import pandas as pd
import sctk
import muon as mu
from datetime import datetime
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

### Set up parameters to save plots and objects

In [2]:
project = 'gut'
species = 'hs'
atribute = 'Elementaite2021_E-MTAB-8901'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

### Load data

In [ ]:
adata = sc.read_h5ad('Processed_data/Elementaite_2021/gut_hs_Elementaite2021_E-MTAB-8901_QC_AM_25112024_121501_raw.h5ad')
adata

## Basic filtering

In [ ]:
sc.pp.filter_cells(adata, min_genes=100)
adata

## Doublet score prediction

In [ ]:
sample_names = adata.obs['Source Name'].unique()

for sample_name in sample_names:
    mask = adata.obs['Source Name'] == sample_name
    sample_adata = adata[mask].copy()

    scrub = scr.Scrublet(sample_adata.X)

    sample_adata.obs['doublet_scores'], sample_adata.obs['predicted_doublets'] = scrub.scrub_doublets()

    adata.obs.loc[mask, 'doublet_scores'] = sample_adata.obs['doublet_scores']
    adata.obs.loc[mask, 'predicted_doublets'] = sample_adata.obs['predicted_doublets']

    #scrub.plot_histogram()

    #plt.show()

In [ ]:
doub_tab = pd.crosstab(adata.obs['Source Name'],adata.obs['predicted_doublets'])
doub_tab.sum()

In [5]:
doub_tab_percentage = doub_tab.div(doub_tab.sum(axis=1), axis=0) * 100

In [ ]:
doub_tab_percentage

In [ ]:
plt.figure(figsize=(15, 7), dpi = 300)
sns.barplot(x=doub_tab_percentage.index, y=doub_tab_percentage[('True')])
plt.ylim(0, 100)
plt.xticks(rotation=90)
plt.xlabel('Source Name')
plt.ylabel('Percentage of Doublets')
plt.title('Percentage of Doublets per Sample')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_scrublet_doublets_{timestamp}_.png")
plt.show()

## Quality Check

+ Calculate QC metrics

In [ ]:
sctk.calculate_qc(adata)

In [7]:
adata.obs['total_counts'] = adata.X.sum(axis=1)

In [8]:
adata.obs['n_genes_by_counts'] = (adata.X > 0).sum(axis=1)

+ Determine which of the cells pass QC individually

In [ ]:
sctk.cellwise_qc(adata)
adata

In [ ]:
adata.obs['cell_passed_qc'].sum()

In [ ]:
adata.uns['scautoqc_ranges']

+ Check the QC parameters (default parameters)

In [ ]:
sctk.default_metric_params_df

+ Cluster cells based on their QC

In [ ]:
metrics_list = ["log1p_n_counts", "log1p_n_genes", "percent_mito", "percent_ribo", "percent_hb"]
sctk.generate_qc_clusters(adata, metrics = metrics_list)
adata

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.embedding(adata, "X_umap_qc", color=["log1p_n_counts", "log1p_n_genes",
                                            "percent_mito", "percent_ribo", "percent_hb", "qc_cluster"], 
                                            color_map="magma_r", frameon=False, ncols=3, size=5, show=False)
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Determine clusters that pass QC

In [ ]:
sctk.clusterwise_qc(adata)
adata

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.embedding(adata, "X_umap_qc", color=["cell_passed_qc", "cluster_passed_qc", 'Source Name'],frameon=False, ncols=3, size=5, show=False,)
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_sctk_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

+ Find the consensus between clusters that pass QC and individual cells that pass QC

In [ ]:
sctk.multi_resolution_cluster_qc(adata, metrics = metrics_list)

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.embedding(adata, "X_umap_qc", color=["cell_passed_qc", 
                                           "cluster_passed_qc", 
                                           "consensus_fraction", 
                                           "consensus_passed_qc", 'Source Name'],frameon=False, ncols=3, color_map="magma_r", size=5, show=False)
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_sctk_multi_resolution_cluster_qc_{timestamp}.png", bbox_inches="tight")
    plt.show()

### Visualize QC metrics before filtering

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.scatterplot(data=adata.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata.obs['total_counts'])) + 1, 15000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - Before filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_counts_vs_genes_before_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(12, 6), ncols=2, gridspec_kw={'width_ratios': [4, 1]})

    sns.violinplot(data=adata.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - Before filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_{var}_before_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [16]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata.obs, x='Source Name')
plt.ylim(0, 120000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - Before filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_number_of_cells_before_filtering_{timestamp}.png", bbox_inches="tight")

### Filter cells that do not pass QC

In [ ]:
adata_filtered = adata[adata.obs['consensus_passed_qc'] == True].copy()
adata_filtered

### Visualize QC metrics after filtering

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=adata_filtered.obs, x='total_counts', y='n_genes_by_counts' , hue ='Source Name', alpha = 0.4, s=4)
plt.legend(title='sample', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(range(0, int(max(adata_filtered.obs['total_counts'])) + 1, 3000),rotation=90, fontsize = 10)
plt.yticks(range(0, int(max(adata_filtered.obs['n_genes_by_counts'])) + 1, 1000),fontsize = 10)
plt.title(f'Counts vs Genes - After filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_counts_vs_genes_after_filtering_{timestamp}.png", bbox_inches="tight")
plt.show()

In [ ]:
variables = 'n_counts', 'log1p_n_counts', 'log1p_n_genes', 'n_genes_by_counts', 'total_counts', 'percent_mito', 'percent_ribo', 'percent_hb', 'doublet_scores'

for var in variables:

    fig, ax = plt.subplots(figsize=(12, 6), ncols=2, gridspec_kw={'width_ratios': [4, 1]})

    sns.violinplot(data=adata_filtered.obs, x='Source Name', y=var, ax=ax[0])
   
    medians = adata_filtered.obs.groupby('Source Name')[var].median()
    
    ax[0].set_title(f'Violin Plot of {var} by Sample - After filtering')
    ax[0].set_xlabel('Sample')
    ax[0].set_ylabel(var)
    ax[0].tick_params(axis='x', rotation=90)

    median_df = pd.DataFrame({'Sample': medians.index, 'Median': medians.values})

    ax[1].axis('off')
    ax[1].table(cellText=median_df.values, colLabels=median_df.columns, loc='center')
    ax[1].set_title('Median Values')
    
    plt.tight_layout()
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_{var}_after_filtering_{timestamp}.png", dpi = 300, bbox_inches="tight")
    plt.show()

In [19]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Source Name')
plt.ylim(0, 12000)
plt.xticks(rotation=90)
plt.title('Number of Cells per Sample - After filtering')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_number_of_cells_after_filtering2_{timestamp}.png", bbox_inches="tight")

## Sex covariate analysis

In [20]:
annot = sc.queries.biomart_annotations(
        "hsapiens",
        ["ensembl_gene_id", "external_gene_name", "start_position", "end_position", "chromosome_name"],
    ).set_index("external_gene_name")

* Chr Y genes calculation

In [21]:
chrY_genes = adata_filtered.var_names.intersection(annot.index[annot.chromosome_name == "Y"])

In [22]:
adata_filtered.obs['percent_chrY'] = np.sum(
    adata_filtered[:, chrY_genes].X, axis = 1) / np.sum(adata_filtered.X, axis = 1) * 100

+ XIST expression analysis

In [23]:
xist_counts = adata_filtered.X[:, adata_filtered.var_names.str.match('XIST')].toarray()

In [24]:
xist_percentage = (np.sum(xist_counts, axis=1) / adata_filtered.obs['total_counts']) * 100

In [25]:
xist_counts_series = pd.Series(xist_counts.squeeze(), index = adata_filtered.obs_names, name = "XIST-counts")
xist_percentage_series = pd.Series(xist_percentage, index=adata_filtered.obs_names, name="percent_XIST")

adata_filtered.obs["XIST-counts"] = xist_counts_series
adata_filtered.obs["XIST-percentage"] = xist_percentage_series

In [ ]:
with plt.rc_context():
    sc.set_figure_params(dpi=300, figsize=(15, 15))
    sc.pl.violin(adata_filtered, ["XIST-counts", "percent_chrY"], jitter = 0.4, groupby = 'Source Name', rotation = 90, show=False)
    plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_sex_covariates_{timestamp}.png", bbox_inches="tight")
    plt.show()

In [27]:
average_xist_counts = adata_filtered.obs.groupby('Source Name')['XIST-counts'].mean()

In [28]:
average_percent_chrY = adata_filtered.obs.groupby('Source Name')['percent_chrY'].mean()

In [29]:
scaled_xist_counts = (average_xist_counts - average_xist_counts.min()) / (average_xist_counts.max() - average_xist_counts.min())
scaled_percent_chrY = (average_percent_chrY - average_percent_chrY.min()) / (average_percent_chrY.max() - average_percent_chrY.min())

In [ ]:
sample_sex = pd.DataFrame({
    'scaled_XIST_counts': scaled_xist_counts,
    'scaled_percent_chrY': scaled_percent_chrY
})
sample_sex['sex'] = np.where(sample_sex['scaled_XIST_counts'] > sample_sex['scaled_percent_chrY'], 'female', 'male')

sample_sex

In [ ]:
adata_filtered.obs = adata_filtered.obs.merge(sample_sex[['sex']], left_on='Source Name', right_index=True, how='left')
adata_filtered

## Calculate cell cycle score

In [34]:
cell_cycle_genes = pd.read_csv('cell_cycle_gene/regev_lab_cell_cycle_genes.txt', header=None)

In [35]:
s_genes = cell_cycle_genes.iloc[0:43]
g2m_genes = cell_cycle_genes.iloc[43:]

In [ ]:
len(s_genes), len(g2m_genes)

In [ ]:
s_genes_in_adata = [gene for gene in s_genes[0] if gene in adata_filtered.var_names]
g2m_genes_in_adata = [gene for gene in g2m_genes[0] if gene in adata_filtered.var_names]
len(s_genes_in_adata), len(g2m_genes_in_adata)

In [38]:
adata_log = ad.AnnData(X = adata_filtered.X,  var = adata_filtered.var, obs = adata_filtered.obs)
sc.pp.normalize_total(adata_log, target_sum = 1e6, exclude_highly_expressed = True)
sc.pp.log1p(adata_log)

In [39]:
sc.tl.score_genes_cell_cycle(adata_log, s_genes = s_genes_in_adata, g2m_genes = g2m_genes_in_adata)

In [40]:
adata_filtered.obs['S_score'] = adata_log.obs['S_score']
adata_filtered.obs['G2M_score'] = adata_log.obs['G2M_score']
adata_filtered.obs['Cell_cycle_phase'] = adata_log.obs['phase']

In [ ]:
plt.figure(figsize=(10, 6), dpi=300)
sns.countplot(data=adata_filtered.obs, x='Cell_cycle_phase')
plt.title('Number of Cells per Cell Cycle Phase')
plt.xlabel('Cell Cycle Phase')
plt.ylabel('Number of Cells')
plt.savefig(f"raw_data/Elmentaite_2021/resulted_plot/{atribute}_cell_cycle_phase_{timestamp}.png", bbox_inches="tight")
plt.show()

### Export anndata object

In [ ]:
adata_filtered.uns['processing_history']

In [45]:
current_history = adata_filtered.uns['processing_history']
new_history = [
     json.dumps(current_history),
     json.dumps({
          'timestamp': timestamp,
          'step': 'filtered cells based on consensus qc generated with sctk (default parameters), added sex covariates and cell cycle phase',
          })
        ]
adata_filtered.uns['processing_history'] = new_history

In [ ]:
adata_filtered.uns['processing_history']

In [ ]:
adata_filtered

In [48]:
adata_filtered.write_h5ad(f"Processed_data/Elementaite_2021/{project}_{species}_{atribute}_QC_filtered_{name}_{timestamp}_{counts}.h5ad")